# Digital Twin Extraction powered by CASCADES

This notebook runs the CASCADES Server (Qwen3-4B with NF4 quantization and CASCADES adapters) and processes your personal Google Takeout data chunks into Neo4j Cypher statements, while simultaneously feeding the structured facts into Hemisphere B for parametric memory consolidation.

It is designed to run on a **T4 GPU** (free tier Google Colab).

## Setup Environment

In [ ]:
!pip install torch torchvision torchaudio
!pip install transformers accelerate bitsandbytes tokenizers
!pip install fastapi uvicorn sse-starlette requests

## Upload Google Takeout Data & Model

1. Mount your Google Drive
2. Ensure `abliteratedqwen3.zip` is in the root of your Google Drive
3. Put your `takeout_chunks_32k` data in your drive
4. Put `CASCADES_Colab_Minimal.zip` in your drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# We will extract the model directly to the fast local Colab NVMe disk
MODEL_PATH = "/content/abliterated"
CHUNKS_DIR = "/content/drive/MyDrive/takeout_chunks_32k"
CYPHER_DIR = "/content/drive/MyDrive/cypher_output"

import os, zipfile, shutil
os.makedirs(CYPHER_DIR, exist_ok=True)

if not os.path.exists(os.path.join(MODEL_PATH, 'config.json')):
    print("Unzipping model to local colab storage (handles nested folders automatically)...")
    os.makedirs(MODEL_PATH, exist_ok=True)
    zip_path = "/content/drive/MyDrive/abliteratedqwen3.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"{zip_path} not found in Google Drive.")
    local_zip = "/content/temp_model.zip"
    if not os.path.exists(local_zip):
        print("Copying zip locally to speed up extraction...")
        shutil.copy(zip_path, local_zip)
    with zipfile.ZipFile(local_zip, "r") as zf:
        members = zf.namelist()
        top_dirs = {m.split("/")[0] for m in members if "/" in m}
        if len(top_dirs) == 1:
            top = top_dirs.pop()
            for member in members:
                rel = os.path.relpath(member, top)
                if rel == ".": continue
                dest = os.path.join(MODEL_PATH, rel)
                if member.endswith("/"): os.makedirs(dest, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(dest), exist_ok=True)
                    with zf.open(member) as src, open(dest, "wb") as dst: dst.write(src.read())
        else:
            zf.extractall(MODEL_PATH)
    os.remove(local_zip)
    print("Model unzipped!")


## Install the CASCADES App

In [ ]:
!mkdir -p /content/CASCADES_REPO
!cp /content/drive/MyDrive/CASCADES_Colab_Minimal.zip /content/CASCADES_REPO/
%cd /content/CASCADES_REPO/
!unzip -o -q CASCADES_Colab_Minimal.zip

import sys
REPO_DIR = "/content/CASCADES_REPO"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

## Start CASCADES Server in Background

This cell actively monitors the background process and will print the server's internal logs immediately if it crashes (OOM, missing module, bad config, etc.).

In [ ]:
import subprocess
import time
import requests

# Start the server using the model on Drive
print(f"Starting CASCADES Server from {MODEL_PATH}...")

cmd = [
    "python", "-m", "app.server",
    "--model_id", MODEL_PATH
]

server_process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT,
    text=True
)

CASCADES_URL = "http://127.0.0.1:8000"
print("Waiting for model to load (checking for crashes)...")

started = False
for i in range(60):
    # 1. Check if the server crashed prematurely
    retcode = server_process.poll()
    if retcode is not None:
        print(f"\n\ud83d\udea8 SERVER CRASHED (Exit code {retcode}) \ud83d\udea8")
        print("--- SERVER ERROR LOGS ---")
        print(server_process.stdout.read())
        break
        
    # 2. Try to ping the server
    try:
        resp = requests.get(f"{CASCADES_URL}/v1/memory/stats", timeout=2)
        if resp.status_code == 200:
            print("\n\u2705 Server is UP!")
            started = True
            break
    except:
        print(".", end="", flush=True)
        time.sleep(5)

if not started and server_process.poll() is None:
    print("\n\u23f3 Server timed out after 5 minutes. Here are the logs to see what it's stuck on:")
    print(server_process.stdout.read())

## Run The Extraction Pipeline

In [ ]:
# Overwrite the script's directories to use the specific paths
import re

script_path = os.path.join(REPO_DIR, 'local_extract_cascades.py')
if os.path.exists(script_path):
    with open(script_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Set new paths
    content = re.sub(
        r'CHUNKS_DIR = r".*?"',
        f'CHUNKS_DIR = r"{CHUNKS_DIR}"',
        content
    )
    content = re.sub(
        r'CYPHER_DIR = r".*?"',
        f'CYPHER_DIR = r"{CYPHER_DIR}"',
        content
    )
    
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("Updated directories in local_extract_cascades.py")
else:
    print(f"Could not find {script_path}")

In [ ]:
!python local_extract_cascades.py

## Manually Check Memory Consolidation Stats

In [ ]:
import requests
resp = requests.get("http://127.0.0.1:8000/v1/memory/stats")
print(resp.json())

## Save Learned Weights to Drive
After processing, the adapter has absorbed the knowledge into parametric memory.

In [ ]:
# The state is maintained in RAM by the server.
# To save it to a .pt file on your Google Drive:
import requests
import shutil
import os
from google.colab import files

print("Ensure you have exported the memory weights!")
# Example of making an imaginary save endpoint request if we added it to server.py:
# requests.post('http://127.0.0.1:8000/v1/memory/save', json={'path':'/content/drive/MyDrive/my_digital_twin_weights.pt'})